# Data Ingestion

In [1]:
import shutil, os
shutil.rmtree("/Users/mandipchhetri/Desktop/AI fellowship 2026/WK15/data/vector_store", ignore_errors=True)
print(os.path.exists("/Users/mandipchhetri/Desktop/AI fellowship 2026/WK15/data/vector_store"))  # must print False

False


In [2]:
from langchain_core.documents import Document

In [3]:
doc = Document(page_content="This is a test document.", metadata={"source": "test"})

In [4]:
doc

Document(metadata={'source': 'test'}, page_content='This is a test document.')

In [5]:
import os

In [6]:
sample_txt = {
    "./data/ingest.txt": "AI was introduced in 2026"
}

for path, content in sample_txt.items():
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        f.write(content)

In [7]:
from langchain_community.document_loaders import TextLoader
loader = TextLoader("./data/ingest.txt",encoding="utf-8")

/var/folders/my/33plt0ns7k16fjlz4qxc2zb00000gn/T/ipykernel_27175/1304111357.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
/Users/mandipchhetri/Desktop/AI fellowship 2026/WK15/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
loader

In [9]:
document = loader.load()

In [10]:
document

[Document(metadata={'source': './data/ingest.txt'}, page_content='AI was introduced in 2026')]

In [11]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader, DirectoryLoader

In [12]:
dir_loader = DirectoryLoader("./data/pdf", glob="*.pdf", loader_cls=PyMuPDFLoader)

In [13]:
pdf_documents = dir_loader.load()

In [14]:
pdf_documents

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-08-16T11:47:29+00:00', 'source': 'data/pdf/Field_Visit_Report_APN (2).pdf', 'file_path': 'data/pdf/Field_Visit_Report_APN (2).pdf', 'total_pages': 19, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-08-16T11:47:29+00:00', 'trapped': '', 'modDate': 'D:20260816114729Z', 'creationDate': 'D:20260816114729Z', 'page': 0}, page_content='TRIBHUVAN UNIVERSITY\nINSTITUTE OF ENGINEERING\nTHAPATHALI CAMPUS\nDEPARTMENT OF ELECTRONICS & COMPUTER ENGINEERING\nFIELD VISIT SUMMARY\nTitle of the field visit/case study: Analysis & Improvement of Process Control & Instrumentation\nSystem at Antarikchya Pratisthan Nepal (Space Foundation Nepal)\nTeam Information\nGroup:\nTeam Members:\n1. Name: Krishna Kandel\nCampus Roll No: THA081BEI014\n2. Name: Mandip Chhetri\nCampus Roll No: THA081BEI018\n3. Name: Nishanta Poudel\nCampus Roll No: THA081BEI025\n4. Nam

### Embeddings and VectorDB

In [15]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [16]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

embedding_manager = EmbeddingManager()


Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11786.57it/s]


Model loaded successfully. Embedding dimension: 384


In [17]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents, chunk_size=500, chunk_overlap=50):
    """ Split documents into smaller chunks
    Args:
        documents: List of Document objects or raw strings.
        chunk_size: Max characters per chunk.
        chunk_overlap: Overlap between chunks.
    Returns:
        List of Document chunks
    """
    if not documents:
        print("No documents to split")
        return []

    # Convert any raw strings to Document objects
    documents = [
        Document(page_content=d, metadata={}) if isinstance(d, str) else d
        for d in documents
    ]

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=['\n\n', '\n', ' ', '']
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    if split_docs:
        print(f"\nExample chunk")
        print(f"Content : {split_docs[0].page_content[:200]}...")
        print(f"Metadata : {split_docs[0].metadata}")

    return split_docs

### VectorStore

In [18]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "./data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG","hnsw:space": "cosine"}  # Use cosine distance
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore


Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [19]:
chunks = split_documents(pdf_documents, chunk_size=500, chunk_overlap=50)

Split 19 documents into 66 chunks

Example chunk
Content : TRIBHUVAN UNIVERSITY
INSTITUTE OF ENGINEERING
THAPATHALI CAMPUS
DEPARTMENT OF ELECTRONICS & COMPUTER ENGINEERING
FIELD VISIT SUMMARY
Title of the field visit/case study: Analysis & Improvement of Proc...
Metadata : {'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-08-16T11:47:29+00:00', 'source': 'data/pdf/Field_Visit_Report_APN (2).pdf', 'file_path': 'data/pdf/Field_Visit_Report_APN (2).pdf', 'total_pages': 19, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-08-16T11:47:29+00:00', 'trapped': '', 'modDate': 'D:20260816114729Z', 'creationDate': 'D:20260816114729Z', 'page': 0}


In [20]:
chunks

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-08-16T11:47:29+00:00', 'source': 'data/pdf/Field_Visit_Report_APN (2).pdf', 'file_path': 'data/pdf/Field_Visit_Report_APN (2).pdf', 'total_pages': 19, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-08-16T11:47:29+00:00', 'trapped': '', 'modDate': 'D:20260816114729Z', 'creationDate': 'D:20260816114729Z', 'page': 0}, page_content='TRIBHUVAN UNIVERSITY\nINSTITUTE OF ENGINEERING\nTHAPATHALI CAMPUS\nDEPARTMENT OF ELECTRONICS & COMPUTER ENGINEERING\nFIELD VISIT SUMMARY\nTitle of the field visit/case study: Analysis & Improvement of Process Control & Instrumentation\nSystem at Antarikchya Pratisthan Nepal (Space Foundation Nepal)\nTeam Information\nGroup:\nTeam Members:\n1. Name: Krishna Kandel\nCampus Roll No: THA081BEI014\n2. Name: Mandip Chhetri\nCampus Roll No: THA081BEI018\n3. Name: Nishanta Poudel\nCampus Roll No: THA081BEI025'),
 Doc

In [21]:
# convert the text to embeddings

texts = [doc.page_content for doc in chunks]

In [22]:
texts

['TRIBHUVAN UNIVERSITY\nINSTITUTE OF ENGINEERING\nTHAPATHALI CAMPUS\nDEPARTMENT OF ELECTRONICS & COMPUTER ENGINEERING\nFIELD VISIT SUMMARY\nTitle of the field visit/case study: Analysis & Improvement of Process Control & Instrumentation\nSystem at Antarikchya Pratisthan Nepal (Space Foundation Nepal)\nTeam Information\nGroup:\nTeam Members:\n1. Name: Krishna Kandel\nCampus Roll No: THA081BEI014\n2. Name: Mandip Chhetri\nCampus Roll No: THA081BEI018\n3. Name: Nishanta Poudel\nCampus Roll No: THA081BEI025',
 'Campus Roll No: THA081BEI025\n4. Name: Prateek Chaulagain\nCampus Roll No: THA081BEI030\n5. Name: Saugat Pokhrel\nCampus Roll No: THA081BEI036\nVisit Details\nOrganization Visited: Antarikchya Pratisthan Nepal (Space Foundation Nepal, APN)\nLocation: Sundhara, Kathmandu, Nepal\nDate of Visit: 10 August 2026\nDepartmental Approval\nName: Asst. Prof. Suwarna Lingden\nName:\nAsst.\nProf.\nUmesh Kanta\nGhimire\nSignature:\nSignature:\nDesignation: Course Instructor\nDesignation: Head of

In [23]:
embedding = embedding_manager.generate_embeddings(texts)

Generating embeddings for 66 texts...


Batches: 100%|██████████| 3/3 [00:00<00:00, 11.41it/s]

Generated embeddings with shape: (66, 384)


In [24]:
vectorstore.add_documents(chunks, embedding)

Adding 66 documents to vector store...
Successfully added 66 documents to vector store
Total documents in collection: 66


### Retriever Pipeline

In [25]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
    
    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            print(results['distances'])
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []
        
rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [26]:
rag_retriever.retrieve("controller architecture running on which firmware?", top_k=5, score_threshold=0)

Retrieving documents for query: 'controller architecture running on which firmware?'
Top K: 5, Score threshold: 0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 129.35it/s]

Generated embeddings with shape: (1, 384)
[[0.5901017189025879, 0.6093093156814575, 0.6308690309524536, 0.6530253887176514, 0.6608455181121826]]
Retrieved 5 documents (after filtering)


[{'id': 'doc_c714d98b_20',
  'content': 'Controllers\nCubeSat: The CubeSat electronics board shown during the visit, labelled “NXT_GEN_CUBUS_DAV”\n(revision V0.2, dated 20/02/2024), deliberately separates the On-Board Computer (OBC) and\nCommunication (Comm) functions onto different microcontrollers rather than combining them on\na single chip. Engineers indicated the main flight microprocessor is an STM32F437AI (STM32F4-\nseries, Arm Cortex-M4), with an STM32WLE5CC handling communications, an STM32WL-',
  'metadata': {'producer': 'pdfTeX-1.40.25',
   'keywords': '',
   'format': 'PDF 1.5',
   'source': 'data/pdf/Field_Visit_Report_APN (2).pdf',
   'file_path': 'data/pdf/Field_Visit_Report_APN (2).pdf',
   'total_pages': 19,
   'subject': '',
   'content_length': 461,
   'creator': 'LaTeX with hyperref',
   'modDate': 'D:20260816114729Z',
   'trapped': '',
   'page': 5,
   'creationdate': '2026-08-16T11:47:29+00:00',
   'creationDate': 'D:20260816114729Z',
   'title': '',
   'moddate':

In [27]:
print(vectorstore.collection.metadata)  # must show 'hnsw:space': 'cosine'
print(vectorstore.collection.count())   # should equal len(chunks), no duplicates

{'hnsw:space': 'cosine', 'description': 'PDF document embeddings for RAG'}
66


## VectorDB Context pipeline with LLM Output

In [28]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [35]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

In [67]:
class GroqLLM:
    def __init__(self, model_name: str = "openai/gpt-oss-120b", api_key: str= ""):
        """
        Initialize Groq LLM
        
        Args:
            model_name: Groq model name (qwen2-72b-instruct, llama3-70b-8192, etc.)
            api_key: Groq API key (or set GROQ_API_KEY environment variable)
        """
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")
        
        if not self.api_key:
            raise ValueError("Groq API key is required. Set GROQ_API_KEY environment variable or pass api_key parameter.")
        
        self.llm = ChatGroq(
            model=self.model_name,
            groq_api_key=self.api_key,
            temperature=0.1,
            max_tokens=1024
        )
        
        print(f"Initialized Groq LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context
        
        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length
            
        Returns:
            Generated response string
        """
        
        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.

Context:
{context}

Question: {question}

Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
        )
        
        # Format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)
        
        try:
            # Generate response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content
            
        except Exception as e:
            return f"Error generating response: {str(e)}"
        
    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting
        
        Args:
            query: User question
            context: Retrieved context
            
        Returns:
            Generated response
        """
        simple_prompt = f"""Based on this context: {context}

Question: {query}

Answer:"""
        
        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"

In [68]:
# Initialize Groq LLM (you'll need to set GROQ_API_KEY environment variable)
try:
    groq_llm = GroqLLM(api_key=os.getenv("GROQ_API_KEY"))
    print("Groq LLM initialized successfully!")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your GROQ_API_KEY environment variable to use the LLM.")
    groq_llm = None

Initialized Groq LLM with model: openai/gpt-oss-120b
Groq LLM initialized successfully!


In [72]:
## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    
    response=groq_llm.generate_response(context=context,query=query)
    return response

In [ ]:
answer=rag_simple("",rag_retriever)
print(answer)

Retrieving documents for query: 'What the field visit about ?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.60it/s]


Generated embeddings with shape: (1, 384)
[[0.6211951971054077, 0.7020987272262573, 0.7291533350944519]]
Retrieved 3 documents (after filtering)
The field visit was a site‑inspection of the river‑level monitoring system.  The team went to the field monitoring location to review the deployed **river‑level Ground Sensor Terminal** (as shown in the GIS/site‑photo in Figure 7) and to examine the supporting facilities – notably the clean‑room and electronics workspaces where the flight hardware and PCBs are handled, tested, and assembled.  In short, the visit was conducted to observe and assess the system components and the physical layout of the monitoring installation.
